In [113]:
pip install groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [114]:
%pip install groq

Note: you may need to restart the kernel to use updated packages.


In [115]:
%pip install -q \
langchain \
langchain-community \
langchain-huggingface \
sentence-transformers \
faiss-cpu \
rank-bm25 \
pypdf \
transformers \
accelerate

Note: you may need to restart the kernel to use updated packages.


In [20]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings loaded successfully")

Embeddings loaded successfully


In [21]:
from langchain_community.vectorstores import FAISS

In [22]:
from langchain_community.llms import HuggingFacePipeline

In [23]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

In [24]:
%pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [25]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from glob import glob


docs = []

pdf_files = glob("data/*.pdf")

print(f"Found {len(pdf_files)} PDF(s)")

for pdf_path in pdf_files:

    print(f"Loading: {os.path.basename(pdf_path)}")

    loader = PyPDFLoader(pdf_path)

    pdf_docs = loader.load()

    for doc in pdf_docs:

        doc.metadata["source"] = os.path.basename(pdf_path)

    docs.extend(pdf_docs)

print(f"Total Pages Loaded: {len(docs)}")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)
split_docs = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

import os

if os.path.exists("vectorstore/faiss_index"):
    
    vectorstore = FAISS.load_local(
        "vectorstore/faiss_index",
        embeddings,
        allow_dangerous_deserialization=True
    )

    print("FAISS index loaded successfully")

else:

    vectorstore = FAISS.from_documents(
        split_docs,
        embeddings
    )

    vectorstore.save_local(
        "vectorstore/faiss_index"
    )

    print("FAISS index created and saved")

faiss_retriever = vectorstore.as_retriever(
    search_kwargs={"k":20}
)

bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 25

retriever = EnsembleRetriever(
    retrievers=[faiss_retriever, bm25_retriever],
    weights=[0.7, 0.3]
)

Found 2 PDF(s)
Loading: EU.pdf
Loading: nist.ai.100-1.pdf
Total Pages Loaded: 192
FAISS index loaded successfully


API KEY

In [26]:
import os
from groq import Groq
from dotenv import load_dotenv


load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

Prompt Building Template

In [27]:
def build_prompt(context, question):

    prompt = f"""
You are an AI Compliance Assistant.

Rules:
1. Answer ONLY from the provided context.
2. Do NOT make up information.
3. If information is not found, say:
   "I could not find this information in the uploaded documents."

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

create rerank

In [28]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Function

In [29]:
def rerank_docs(query, docs):

    pairs = [
        (query, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    return [
        doc
        for score, doc
        in ranked_docs[:10]
    ]


Guardrail Function

In [30]:
def check_context(docs):

    if len(docs) == 0:
        return False

    context = " ".join(
        [
            doc.page_content
            for doc in docs
        ]
    )

    
    if len(context.strip()) < 100:
        return False

    return True

Generate Answer func

In [45]:
def generate_answer(question):

    # Retrieve documents
    retrieved_docs = retriever.invoke(question)

    # Rerank documents
    reranked_docs = rerank_docs(
        question,
        retrieved_docs
    )

    # Guardrail
    if not check_context(reranked_docs):

        return {
            "answer":
            "I could not find relevant information in the uploaded documents.",
            "sources":[]
        }

    # Create context
    top_docs = reranked_docs[:5]

    context = "\n\n".join(
       [
          doc.page_content
          for doc in top_docs
       ]
    )

    # Build prompt
    prompt = build_prompt(
        context,
        question
    )

    # Generate answer
    response = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[
           {
              "role":"user",
              "content":prompt
           }
        ],
        temperature=0.1
    )

    answer = response.choices[0].message.content
    

    import re

    answer = re.sub(
        r"<think>.*?</think>",
        "",
        answer,
        flags=re.DOTALL
    ).strip()
    

    return {
    "answer": answer,
    "sources": [reranked_docs[0]]
}
    

Testing RAG Pipeline

In [49]:
response = generate_answer(
    "What are risks, impacts and harms in AI RMF?"
)

print(response["answer"])

In the **NIST AI Risk Management Framework (AI RMF) 1.0**, **risks**, **impacts**, and **harms** are defined and addressed as follows:  

1. **Risks**:  
   - Risk is a **composite measure** of the probability of an event occurring and the magnitude of its consequences.  
   - It includes both **opportunities (positive outcomes)** and **threats (negative outcomes)**.  
   - Risks are prioritized based on context, such as systems interacting with humans, using sensitive data (e.g., personally identifiable information), or having direct/indirect impacts on humans.  
   - Unacceptable risks (e.g., imminent severe harms or catastrophic risks) require halting development/deployment until risks are managed.  

2. **Impacts**:  
   - Impacts are the **consequences** of AI systems, which can be **positive, negative, or both**.  
   - They may affect individuals, communities, organizations, or ecosystems.  
   - Impacts vary by context, including differences in how groups or sub-groups (e.g., n

In [50]:
for doc in response["sources"]:
    print(
        doc.metadata["source"],
        doc.metadata["page"]
    )

nist.ai.100-1.pdf 8


Citation

In [51]:
response = generate_answer(
    "What is the full name of the NIST AI Risk Management Framework?"
)

print(response["answer"])

print("\nSources:\n")

for doc in response["sources"]:

    print(
        f"""
File:
{doc.metadata.get('source','Unknown')}

Page:
{doc.metadata.get('page','Unknown')}
"""
    )

The full name of the NIST AI Risk Management Framework is the **Artificial Intelligence Risk Management Framework (AI RMF 1.0)**.

Sources:


File:
nist.ai.100-1.pdf

Page:
0

